In [3]:
import re
from collections import Counter

posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ 자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강. 문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    " 여러 공백과\n\n\n줄바꿈이 많은 텍스트 ",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]

URL_PATTERN = re.compile(r"https?://\S+")
HTML_PATTERN = re.compile(r"<[^>]+>")
EMAIL_PATTERN = re.compile(r"[\w.+-]+@[\w.-]+\.\w+")
PHONE_PATTERN = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")
MENTION_HASHTAG_PATTERN = re.compile(r"[@#]\w+")
JAMO_PATTERN = re.compile(r"[ㄱ-ㅣ]+")
SPACE_PATTERN = re.compile(r"\s+")
HASHTAG_PATTERN = re.compile(r"#([가-힣A-Za-z0-9]+)")


def clean_post(post: str) -> str:
    post = URL_PATTERN.sub(" ", post)
    post = HTML_PATTERN.sub("", post)
    post = EMAIL_PATTERN.sub("[이메일]", post)
    post = PHONE_PATTERN.sub("[전화]", post)
    post = MENTION_HASHTAG_PATTERN.sub(" ", post)
    post = JAMO_PATTERN.sub("", post)
    post = SPACE_PATTERN.sub(" ", post)

    return post.strip()


def extract_hashtags(post: str) -> list[str]:
    return HASHTAG_PATTERN.findall(post)


def analyze_posts(posts: list[str]) -> dict:
    cleaned_posts = [clean_post(post) for post in posts]

    avg_length = round(
        sum(len(post) for post in cleaned_posts) / len(cleaned_posts),
        2
    )

    hashtags: list[str] = []
    for post in posts:
        hashtags.extend(extract_hashtags(post))

    hashtag_counts = dict(Counter(hashtags).most_common())

    masked_count = 0

    for post in posts:
        _, email_n = EMAIL_PATTERN.subn("[이메일]", post)
        post_after_email = EMAIL_PATTERN.sub("[이메일]", post)

        _, phone_n = PHONE_PATTERN.subn("[전화]", post_after_email)

        masked_count += email_n + phone_n

    return {
        "posts_n": len(posts),
        "avg_length_after_clean": avg_length,
        "hashtag_counts": hashtag_counts,
        "masked_count": masked_count,
    }